# Notebook 4 — PyTorch Custom Dataset & DataLoader

This notebook builds the PyTorch data pipeline from the CSV splits:
1. **Read CSVs** — load `train.csv`, `val.csv`, `test.csv` from `data/splits/`
2. **Custom `Dataset` class** — inherits `torch.utils.data.Dataset`, loads images via OpenCV
3. **Transforms** — training uses augmentation; val/test use normalize-only
4. **`DataLoader`** — batching, shuffling, and parallel loading

> **Prerequisites:** Notebook 2 must have been run to generate `data/processed/` and `data/splits/`.

This pipeline is the PyTorch equivalent of the Keras custom generator (Notebook 2) and produces identically structured batches for fair CNN vs ViT comparison.

In [ ]:


import os
import cv2
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

## PyTorch version
1.  **Custom `Dataset` Class:** You'll build our own dataset class from scratch by inheriting from `torch.utils.data.Dataset`. This gives you full control and a deep understanding of the data pipeline.
2.  **In-built `ImageFolder` Utility:** You'll use the convenient `torchvision.datasets.ImageFolder` class, which automatically handles data from a standard directory structure.
3.  **The `DataLoader`:** You'll wrap both of our datasets in a `DataLoader`, PyTorch's engine for efficient batching, shuffling, and parallelized data loading.

1. Read CSV (train/val/test)

2. Custom Dataset class
   - load image from path
   - apply transforms

3. Define transforms
   - train → normalize + augmentation
   - val/test → normalize only

4. DataLoader
   - batching
   - shuffling (train only)

In [ ]:
train_df = pd.read_csv("../data/splits/train.csv")
val_df   = pd.read_csv("../data/splits/val.csv")
test_df  = pd.read_csv("../data/splits/test.csv")

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    transforms.ToTensor(),  # converts to [0,1]
])

In [ ]:
val_test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

In [ ]:
class LandDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['filepath']
        label = self.df.iloc[idx]['label']

        # Load image
        img = cv2.imread(img_path)
        img = cv2.resize(img, (64, 64))

        # Convert BGR → RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
train_dataset = LandDataset(train_df, transform=train_transforms)
val_dataset   = LandDataset(val_df, transform=val_test_transforms)
test_dataset  = LandDataset(test_df, transform=val_test_transforms)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)  # [B, C, H, W]
print("Label batch shape:", labels.shape)